# 2. Translation and merge

Takes the long files built by `Compendium_1_Long_Files.ipynb` and produces one
English file per chapter:

1. **Translate** `merged longfiles_AR\<Chapter>_AR.xlsx` into English.
2. **Append** `merged longfiles_EN\<Chapter>_EN.xlsx` if that chapter also had
   English questionnaires - those rows are already English and need no
   translation.
3. **Save** the combined result as `COMPENDIUM-ARAB SOCIETY\<Chapter>_EN.xlsx`.
4. **Calculate** the sex ratio and the age-group percentages on that combined
   file, so a country that only submitted in English is included.

The combined file is written to a third location and rebuilt from scratch each
run, so `merged longfiles_EN` stays purely what notebook 1 put there and
re-running can never append the same rows twice.

Anything the dictionary has no entry for passes through untranslated and is
listed at the end, ready for `export_untranslated()` -> Claude Code ->
`update_dictionary()`.


In [ ]:
"""
CELL: Imports and logging setup.
"""
import difflib
import logging
import re
from collections import defaultdict
from pathlib import Path

import pandas as pd

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("compendium")


## Config / paths


In [ ]:
"""
CELL: Configuration - paths and chapters.
"""
DATA_COLLECTOR_PATH = Path(r"C:\Users\RSHIRINI\OneDrive - United Nations\Desktop\DSS\DATA COLLECTOR")
TRANSLATION_DICT_PATH = DATA_COLLECTOR_PATH / "translation dict.xlsx"
COMPENDIUM_PATH = Path(r"C:\Users\RSHIRINI\OneDrive - United Nations\Desktop\DSS\COMPENDIUM-ARAB SOCIETY")

# Questionnaire folders are named <prefix><LANGUAGE>. The suffix IS the language,
# so nothing here has to list them - add a folder and it is picked up.
QUESTIONNAIRE_PREFIX = "datacollector_received_quest_"

# One long file per chapter per language lands here.
def long_file_folder(language):
    return COMPENDIUM_PATH / f"merged longfiles_{language}"

# Leave CHAPTERS as None to process every chapter found on disk. Set an explicit
# list to restrict one run, e.g. CHAPTERS = ["Poverty"].
CHAPTERS = None

LANGUAGES = ["AR", "EN"]

# The language the questionnaires mostly arrive in, and the one we translate to.
SOURCE_LANGUAGE = "AR"
TARGET_LANGUAGE = "EN"


def discover_chapters():
    """Chapters that notebook 1 produced a long file for, in either language."""
    names = set()
    for language in LANGUAGES:
        folder = long_file_folder(language)
        if not folder.exists():
            continue
        suffix = f"_{language}.xlsx"
        for path in folder.glob(f"*{suffix}"):
            names.add(path.name[: -len(suffix)])
    return sorted(names)


def chapters_to_process():
    """CHAPTERS if it was set, otherwise whatever notebook 1 produced."""
    if CHAPTERS:
        return list(CHAPTERS)
    found = discover_chapters()
    logger.info(f"Chapters with a long file: {found}")
    if not found:
        logger.warning("No long files found - run notebook 1 first.")
    return found

# The dictionary lives outside the repo, so git cannot track it. Every update
# also writes this CSV snapshot inside the repo: the .xlsx stays the working
# copy, the CSV is the versioned record - and CSV diffs cleanly where .xlsx
# does not. Commit it after a dictionary change to keep the history.
DICTIONARY_SNAPSHOT_PATH = Path(r"c:\Users\RSHIRINI\OneDrive - United Nations\Desktop\DSS\COMPENDIUM-ARAB SOCIETY\codes\translation_dict_snapshot.csv")


## Load the translation dictionary

One dictionary, Arabic to English - no reverse is built, because the
questionnaires arrive in Arabic and English is what we translate into.


In [ ]:
"""
CELL: Load translation dict.xlsx - one dictionary, Arabic to English.
"""


def load_dictionary():
    """Reads translation dict.xlsx into:

      DICTIONARY_AR_TO_EN  (column_map, value_map)
          column_map = {Arabic column name: English column name}
          value_map  = {Arabic column name: {Arabic value: English value}}

      ENGLISH_VOCABULARY  (column_names, values_by_column)
          the English column names and values the file knows about.

    Only the Arabic-to-English direction is built, because the questionnaires
    arrive in Arabic and English is what we translate into.

    ENGLISH_VOCABULARY is not a reverse dictionary - it maps nothing. It is just
    the list of correct English spellings, so an English questionnaire's
    misspelled labels can be fuzzy-matched against the right words too.
    """
    dict_df = pd.read_excel(TRANSLATION_DICT_PATH, engine="openpyxl")

    column_map, value_map = {}, {}
    for arabic_column in dict_df["col_ar"].dropna().unique():
        rows = dict_df[dict_df["col_ar"] == arabic_column]
        column_map[arabic_column] = rows["col_en"].iloc[0]
        value_map[arabic_column] = {
            arabic: english
            for arabic, english in zip(rows["val_ar"], rows["val_en"])
            if pd.notna(arabic)
        }

    english_columns = {}
    english_values = {}
    for english_column in dict_df["col_en"].dropna().unique():
        rows = dict_df[dict_df["col_en"] == english_column]
        english_columns[english_column] = english_column
        english_values[english_column] = {
            str(v): str(v) for v in rows["val_en"].dropna().unique()
        }

    chapter_rows = dict_df[dict_df["col_en"] == "Chapter"]
    chapter_to_arabic = dict(zip(chapter_rows["val_en"], chapter_rows["val_ar"]))

    return (column_map, value_map), (english_columns, english_values), chapter_to_arabic


DICTIONARY_AR_TO_EN, ENGLISH_VOCABULARY, CHAPTER_TO_ARABIC = load_dictionary()


def vocabulary(language):
    """The known column names and values for a language, as
    (column_names, values_by_column) - what misspellings are matched against."""
    return DICTIONARY_AR_TO_EN if language == "AR" else ENGLISH_VOCABULARY


def column_name(arabic_name, language):
    """One of the pipeline's own column names, spelled for the given language."""
    arabic_to_english, _ = DICTIONARY_AR_TO_EN
    return arabic_name if language == "AR" else arabic_to_english[arabic_name]


arabic_columns, _ = DICTIONARY_AR_TO_EN
logger.info(f"Dictionary loaded: {len(arabic_columns)} column names, Arabic -> English")


## `translate()`

Swaps every Arabic value for its English equivalent, then renames the column.
Anything the dictionary has no entry for is left exactly as it is.


In [ ]:
"""
CELL: translate() - Arabic to English, using the one dictionary.
"""


def translate(table):
    """Swaps every Arabic value for its English equivalent, then renames the
    column to its English name.

    Values are replaced first and the column renamed second, because the value
    lookup is keyed by the column's ORIGINAL Arabic name - renaming first would
    lose it. Anything the dictionary has no entry for is left exactly as it is,
    and is picked up afterwards by find_untranslated().
    """
    column_map, value_map = DICTIONARY_AR_TO_EN

    table = table.copy()
    for column in list(table.columns):
        if column in value_map:
            table[column] = table[column].replace(value_map[column])
        if column in column_map:
            table = table.rename(columns={column: column_map[column]})
    return table


def looks_arabic(text):
    """True if the text contains at least one Arabic letter. U+0600-U+06FF is
    the Arabic Unicode block; English text has nothing in it."""
    return any("\u0600" <= character <= "\u06ff" for character in str(text))


## Finding and filling the dictionary's gaps

### Letting Claude Code close the gaps for you

You do not have to shuttle the file back and forth. Ask once:

> **"run the pipeline and fill any dictionary gaps"**

Claude Code runs the notebook, translates whatever the dictionary did not know,
writes it back into `translation dict.xlsx` (taking a backup first), re-runs to
confirm the gap list is empty, and reports what it added. The protocol is
recorded in `CLAUDE.md`, so it does not need explaining again each session.

Doing it by hand is still the same three calls: `export_untranslated(REPORTS)`
-> fill in the blank column -> `update_dictionary(filled)`.


In [ ]:
"""
CELL: find_untranslated() - what the dictionary could not translate.
"""


def find_untranslated(arabic_table, translated_table):
    """Values that came out of translate() unchanged and are still Arabic.

    A value with no dictionary entry is copied through untouched, so comparing
    the table before and against after finds them exactly. Values already in
    Latin script are not gaps - plenty of Arabic questionnaires cite their
    source in English ("MICS 2022", a URL) and those are correct as they stand.
    """
    column_map, _ = DICTIONARY_AR_TO_EN
    gaps = []

    for arabic_column, english_column in zip(arabic_table.columns, translated_table.columns):
        before = arabic_table[arabic_column]
        after = translated_table[english_column]
        pairs = pd.DataFrame({"val_ar": before, "val_en": after}).dropna()

        for (arabic_value, english_value), count in pairs.groupby(["val_ar", "val_en"]).size().items():
            if str(arabic_value).strip() != str(english_value).strip():
                continue                      # translated fine
            if not looks_arabic(arabic_value):
                continue                      # already English
            gaps.append({
                "col_ar": arabic_column, "col_en": english_column,
                "val_ar": arabic_value, "val_en": None, "rows": count,
            })
    return gaps


def export_untranslated(reports, file_name="untranslated_values.xlsx"):
    """Writes every gap found during the run to one Excel file, shaped like
    translation dict.xlsx so a filled-in row can go straight back into it."""
    rows = [gap for report in reports for gap in report["untranslated"]]
    if not rows:
        logger.info("Nothing untranslated - the dictionary covered every value.")
        return pd.DataFrame()

    gaps = (pd.DataFrame(rows)
            .drop_duplicates(subset=["col_ar", "val_ar"])
            .sort_values(["col_en", "val_ar"])
            .reset_index(drop=True))
    path = COMPENDIUM_PATH / file_name
    gaps.to_excel(path, index=False, engine="openpyxl")
    logger.info(f"Saved {path.name}: {len(gaps):,} value(s) with no dictionary entry. "
                f"Fill in val_en, then call update_dictionary().")
    return gaps


def update_dictionary(filled, backup=True):
    """Appends reviewed translations to translation dict.xlsx.

    `filled` is the exported table with val_en filled in (a DataFrame, or a path
    to the saved file). Rows missing either side are skipped, and an Arabic
    value the dictionary already has is left alone - so running this twice
    changes nothing the second time. A timestamped backup is written first,
    because this edits the project's source of truth.
    """
    if not isinstance(filled, pd.DataFrame):
        filled = pd.read_excel(filled, engine="openpyxl")

    needed = ["col_ar", "val_ar", "col_en", "val_en"]
    missing = [c for c in needed if c not in filled.columns]
    if missing:
        raise ValueError(f"missing column(s) {missing}; expected {needed}")

    new_rows = filled[needed].dropna()
    new_rows = new_rows[(new_rows["val_ar"].astype(str).str.strip() != "")
                        & (new_rows["val_en"].astype(str).str.strip() != "")]
    if new_rows.empty:
        logger.warning("No completed rows to add - is val_en filled in?")
        return None

    dictionary = pd.read_excel(TRANSLATION_DICT_PATH, engine="openpyxl")
    already_there = set(zip(dictionary["col_ar"], dictionary["val_ar"]))
    to_add = new_rows[~new_rows.apply(
        lambda r: (r["col_ar"], r["val_ar"]) in already_there, axis=1)]
    if to_add.empty:
        logger.info("Every row is already in the dictionary - nothing to add.")
        return dictionary

    if backup:
        stamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
        backup_path = TRANSLATION_DICT_PATH.with_name(
            f"{TRANSLATION_DICT_PATH.stem} backup {stamp}.xlsx")
        dictionary.to_excel(backup_path, index=False, engine="openpyxl")
        logger.info(f"Backed up the dictionary to {backup_path.name}")

    updated = pd.concat([dictionary, to_add.reindex(columns=dictionary.columns)],
                        ignore_index=True)
    updated.to_excel(TRANSLATION_DICT_PATH, index=False, engine="openpyxl")
    # utf-8-sig so the Arabic opens correctly if the snapshot is viewed in Excel.
    updated.to_csv(DICTIONARY_SNAPSHOT_PATH, index=False, encoding="utf-8-sig")
    logger.info(f"Snapshot written to {DICTIONARY_SNAPSHOT_PATH.name} - commit it "
                f"to keep the dictionary's history")
    logger.info(f"Added {len(to_add):,} row(s) to {TRANSLATION_DICT_PATH.name} "
                f"({len(dictionary):,} -> {len(updated):,}). Re-run to use them.")
    return updated


def calculated_labels():
    """Every label this notebook INVENTS, as (English column, English value).

    These come from the calculations, not from any questionnaire, so the
    dictionary can only ever learn them from here. Derived from the same
    constants the calculations use, so adding a calculation cannot leave this
    list behind.
    """
    labels = [("Indicator", SEX_RATIO_TITLE), ("Indicator", AGE_SHARE_TITLE)]
    labels += [("Age Group", group) for group in AGE_GROUPS]
    return labels


def check_calculated_labels():
    """Which invented labels the dictionary does not know yet.

    Returned in the gap shape used elsewhere, except that here it is the ARABIC
    side that is blank: the English is what this notebook chose, and the Arabic
    is what has to be supplied before notebook 3 can render these rows.
    """
    dictionary = pd.read_excel(TRANSLATION_DICT_PATH, engine="openpyxl")
    known = set(zip(dictionary["col_en"].astype(str).str.strip(),
                    dictionary["val_en"].astype(str).str.strip()))

    column_map, _ = DICTIONARY_AR_TO_EN
    english_to_arabic_column = {en: ar for ar, en in column_map.items()}

    missing = []
    for english_column, label in calculated_labels():
        if (english_column, str(label).strip()) in known:
            continue
        missing.append({
            "col_ar": english_to_arabic_column.get(english_column, english_column),
            "col_en": english_column,
            "val_ar": None,          # <- to be filled in
            "val_en": label,
            "rows": None,
        })
    return pd.DataFrame(missing)


def export_calculated_labels(file_name="new_labels_to_translate.xlsx"):
    """Writes the invented labels the dictionary does not know to their own
    file, kept separate from the questionnaire gaps because the blank column is
    the other one."""
    missing = check_calculated_labels()
    if missing.empty:
        logger.info("The dictionary knows every label the calculations invent.")
        return missing
    path = COMPENDIUM_PATH / file_name
    missing.to_excel(path, index=False, engine="openpyxl")
    logger.info(f"Saved {path.name}: {len(missing)} invented label(s) with no Arabic. "
                f"Fill in val_ar, then call update_dictionary().")
    return missing


## `build_chapter()`

Translate the Arabic long file, append the native English one if it exists, and
save the combined result. Every step is logged, so the run output shows
exactly which chapters had an English file appended.


In [ ]:
"""
CELL: build_chapter() - translate, append the native English file, save.
"""


def build_chapter(chapter):
    """Produces COMPENDIUM-ARAB SOCIETY/<chapter>_EN.xlsx and reports what went
    into it.

    Returns a dict describing the run, so the cell below can print one table
    showing which chapters had an English long file appended and which did not.
    """
    arabic_path = long_file_folder("AR") / f"{chapter}_AR.xlsx"
    english_path = long_file_folder("EN") / f"{chapter}_EN.xlsx"
    output_path = COMPENDIUM_PATH / f"{chapter}_EN.xlsx"

    report = {"chapter": chapter, "translated_rows": 0, "appended_rows": 0,
              "total_rows": 0, "appended_from": None, "untranslated": []}

    parts = []

    if arabic_path.exists():
        arabic_table = pd.read_excel(arabic_path, engine="openpyxl")
        translated = translate(arabic_table)
        parts.append(translated)
        report["translated_rows"] = len(translated)
        logger.info(f"  {chapter}: translated {len(translated):,} row(s) from "
                    f"merged longfiles_AR\\{arabic_path.name}")
        report["untranslated"] = find_untranslated(arabic_table, translated)
    else:
        logger.info(f"  {chapter}: no Arabic long file")

    if english_path.exists():
        english_table = pd.read_excel(english_path, engine="openpyxl")
        parts.append(english_table)
        report["appended_rows"] = len(english_table)
        report["appended_from"] = f"merged longfiles_EN\\{english_path.name}"
        logger.info(f"  {chapter}: APPENDED {len(english_table):,} row(s) from "
                    f"{report['appended_from']} (already English, not translated)")
    else:
        logger.info(f"  {chapter}: no English long file to append")

    if not parts:
        logger.warning(f"  {chapter}: nothing to build - run notebook 1 first")
        return report

    # Concatenate, not merge side-by-side: both parts are the same long shape,
    # so this stacks the English-sourced rows under the translated ones.
    combined = pd.concat(parts, ignore_index=True)
    combined.to_excel(output_path, index=False, engine="openpyxl")
    report["total_rows"] = len(combined)
    logger.info(f"  {chapter}: saved {output_path.name} ({len(combined):,} rows)")
    return report


## Run - translate, append, save


In [ ]:
"""
CELL: Main run - build every chapter's combined English file.
"""
print(f"Translating {SOURCE_LANGUAGE} -> {TARGET_LANGUAGE}")
print(f"  reading   {COMPENDIUM_PATH}\\merged longfiles_AR")
print(f"  appending {COMPENDIUM_PATH}\\merged longfiles_EN")
print(f"  writing   {COMPENDIUM_PATH}\\<Chapter>_EN.xlsx\n")

REPORTS = []
chapters = chapters_to_process()
total_chapters = len(chapters)
for i, chapter in enumerate(chapters, start=1):
    bar = "#" * i + "-" * (total_chapters - i)
    print(f"[{bar}] chapter {i}/{total_chapters}: {chapter}")
    REPORTS.append(build_chapter(chapter))

print("\n" + "=" * 78)
print("WHAT WENT INTO EACH FILE")
print("=" * 78)
print(f"{'Chapter':<12}{'translated AR':>15}{'appended EN':>14}{'total':>12}   appended from")
for r in REPORTS:
    appended = r["appended_from"] or "-  (no English long file)"
    print(f"{r['chapter']:<12}{r['translated_rows']:>15,}{r['appended_rows']:>14,}"
          f"{r['total_rows']:>12,}   {appended}")

with_english = [r["chapter"] for r in REPORTS if r["appended_rows"]]
print(f"\nChapters that had an English long file appended: "
      f"{with_english if with_english else 'none'}")

gap_count = len({(g["col_ar"], g["val_ar"]) for r in REPORTS for g in r["untranslated"]})
print(f"Distinct values with no dictionary entry: {gap_count}")
if gap_count:
    print("Run export_untranslated(REPORTS), ask Claude Code to fill in val_en,")
    print("then update_dictionary() - and run this cell again.")


## Calculations

Both run on the **combined** English file, so a country that only submitted in
English is included.

### Where the numbers come from

Two indicators carry population by sex and age - `Population size by
nationality` (sliced Total / Nationals / Non-nationals) and `Population size by
area` (Total / Urban / Rural). **They describe the same people**, so the code:

- takes only the **total** slice of whichever it uses, never summing the parts,
  which would count some people two or three times;
- prefers the nationality indicator, which covers far more countries, and falls
  back to the area one only for a country the first does not have at all;
- never adds the two indicators together.

### Sex ratio, 2010-2025 (per 100 females)

`male population / female population x 100`, using each country-year's
all-ages total. Where that reported total disagrees with the sum of its own age
bands by more than 1%, the source figures contradict themselves - those cases
are logged as warnings rather than silently used.

### Percentage of population by age group and by sex, 2024

The five-year bands are collapsed into `<15`, `15-64` and `65+`, each expressed
as a share of that sex's total. The denominator is the sum of the three groups,
not the reported `Age Total`, so a country missing a band still adds to 100%.
`Age unknown` is excluded - it cannot be placed in a band.


In [ ]:
"""
CELL: The two calculations, on the combined English file.
"""

AGE_GROUPS = {
    "<15 years": ["0-4 years", "5-9 years", "10-14 years"],
    "15-64 years": ["15-19 years", "20-24 years", "25-29 years", "30-34 years",
                    "35-39 years", "40-44 years", "45-49 years", "50-54 years",
                    "55-59 years", "60-64 years"],
    "65+ years": ["65-69 years", "70-74 years", "75+ years"],
}

SEX_RATIO_TITLE = "Sex ratio, 2010-2025 (per 100 females)"
AGE_SHARE_TITLE = "Percentage of population by age group and by sex, 2024"


def to_number(value):
    """Parse one Value cell into a float, or None if there is no number in it.

    These files store figures as text more often than as numbers, and not
    consistently: ' 701 956 ' uses spaces as thousand separators, some cells use
    commas, some carry a non-breaking space, and a few hold '-' for "no data".
    float() alone fails on all but the plainest of them.
    """
    if pd.isna(value):
        return None
    text = str(value).replace("\xa0", " ").replace(",", "").strip()
    text = re.sub(r"\s+", "", text)
    if text in ("", "-", "--", "..", "..."):
        return None
    try:
        return float(text)
    except ValueError:
        return None


def population_base(table):
    """Population counts as country / year / sex / age band, with no row counted
    twice. See the notes above for why the total slice is taken and why the two
    indicators are never added together."""
    frames = []
    for indicator, slice_column, slice_value in [
        ("Population size by nationality", "Nationality", "Nationality Total"),
        ("Population size by area", "Area", "Area Total"),
    ]:
        if slice_column not in table.columns:
            continue
        part = table[(table["Indicator"] == indicator)
                     & (table[slice_column] == slice_value)].copy()
        part["source_indicator"] = indicator
        frames.append(part)

    if not frames:
        return pd.DataFrame(columns=["Country", "Sex", "Age Group", "Year", "number"])

    combined = pd.concat(frames, ignore_index=True)
    combined["number"] = combined["Value"].map(to_number)
    combined = combined[combined["number"].notna()]

    preferred = "Population size by nationality"
    covered = set(combined.loc[combined["source_indicator"] == preferred, "Country"])
    keep = (combined["source_indicator"] == preferred) | (~combined["Country"].isin(covered))
    return combined[keep][["Country", "Sex", "Age Group", "Year", "number"]]


def calculate_sex_ratio(base, first_year=2010, last_year=2025):
    """One row per country and year: males per 100 females, all ages."""
    totals = base[(base["Age Group"] == "Age Total")
                  & (base["Sex"].isin(["Male", "Female"]))
                  & (base["Year"].astype(int).between(first_year, last_year))]

    wide = totals.pivot_table(index=["Country", "Year"], columns="Sex",
                              values="number", aggfunc="first")
    for needed in ("Male", "Female"):
        if needed not in wide.columns:
            logger.warning(f"Sex ratio: no '{needed}' rows - cannot calculate")
            return pd.DataFrame(columns=["Country", "Year", "Value"])
    wide = wide.dropna(subset=["Male", "Female"])
    wide = wide[wide["Female"] > 0]

    result = (wide["Male"] / wide["Female"] * 100).round(1).reset_index()
    result.columns = ["Country", "Year", "Value"]

    # Cross-check the reported all-ages total against the sum of the bands.
    bands = [b for group in AGE_GROUPS.values() for b in group]
    summed = (base[base["Age Group"].isin(bands)]
              .groupby(["Country", "Year", "Sex"])["number"].sum(min_count=1))
    reported = totals.set_index(["Country", "Year", "Sex"])["number"]
    shared = summed.index.intersection(reported.index)
    if len(shared):
        gap = (summed.loc[shared] - reported.loc[shared]).abs() / reported.loc[shared].abs()
        bad = gap[gap > 0.01]
        if len(bad):
            logger.warning(
                f"Sex ratio: {len(bad)} country/year/sex cell(s) where the reported "
                f"all-ages total and the sum of the age bands disagree by more than 1% "
                f"- the source figures contradict themselves:")
            for (country, year, sex), value in bad.sort_values(ascending=False).head(8).items():
                logger.warning(f"    {country} {year} {sex}: off by {value:.0%}")

    logger.info(f"Sex ratio: {len(result)} country/year value(s)")
    return result


def calculate_age_group_percentages(base, year=2024):
    """The percentage of each sex's population in each of the three age groups."""
    band_to_group = {band: group for group, bands in AGE_GROUPS.items() for band in bands}

    rows = base[(base["Year"].astype(int) == year)
                & (base["Sex"].isin(["Male", "Female"]))
                & (base["Age Group"].isin(band_to_group))].copy()
    if rows.empty:
        logger.warning(f"Age groups: no usable population rows for {year}")
        return pd.DataFrame(columns=["Country", "Sex", "Age Group", "Value"])

    rows["group"] = rows["Age Group"].map(band_to_group)
    by_group = rows.groupby(["Country", "Sex", "group"])["number"].sum(min_count=1)
    per_sex = by_group.groupby(["Country", "Sex"]).sum()

    result = (by_group / per_sex * 100).round(1).reset_index()
    result.columns = ["Country", "Sex", "Age Group", "Value"]
    result = result[result["Value"].notna()]

    check = result.groupby(["Country", "Sex"])["Value"].sum().round(0)
    off = check[(check - 100).abs() > 1]
    if len(off):
        logger.warning(f"Age groups: {len(off)} country/sex group(s) not summing to 100%:")
        for (country, sex), total in off.items():
            logger.warning(f"    {country} {sex}: {total}%")
    else:
        logger.info(f"Age groups: every country/sex adds to 100% ({len(check)} checked)")

    logger.info(f"Age groups {year}: {len(result)} value(s), "
                f"{result['Country'].nunique()} countries")
    return result


def add_calculations(chapter, year_for_age_groups=2024):
    """Appends both calculated indicators to <chapter>_EN.xlsx.

    Rows from a previous run are removed first, so this is safe to repeat even
    though the run cell above already rebuilds the file from scratch.
    """
    path = COMPENDIUM_PATH / f"{chapter}_EN.xlsx"
    if not path.exists():
        logger.info(f"{chapter}: no {path.name} - nothing to calculate on")
        return None

    table = pd.read_excel(path, engine="openpyxl")
    if "Indicator" not in table.columns:
        logger.warning(f"{chapter}: no Indicator column, skipping calculations")
        return table

    already = table["Indicator"].isin([SEX_RATIO_TITLE, AGE_SHARE_TITLE])
    if already.any():
        logger.info(f"{chapter}: removing {already.sum():,} row(s) from a previous run")
        table = table[~already]

    base = population_base(table)
    if base.empty:
        logger.info(f"{chapter}: no population rows - nothing to calculate")
        return table

    chapter_value = table["Chapter"].dropna().iloc[0] if "Chapter" in table.columns else None

    ratio = calculate_sex_ratio(base)
    ratio_rows = pd.DataFrame({
        "Indicator": SEX_RATIO_TITLE, "Country": ratio["Country"],
        "Year": ratio["Year"], "Value": ratio["Value"], "Chapter": chapter_value,
    })

    shares = calculate_age_group_percentages(base, year=year_for_age_groups)
    share_rows = pd.DataFrame({
        "Indicator": AGE_SHARE_TITLE, "Country": shares["Country"],
        "Sex": shares["Sex"], "Age Group": shares["Age Group"],
        "Year": year_for_age_groups, "Value": shares["Value"], "Chapter": chapter_value,
    })

    combined = pd.concat([table, ratio_rows, share_rows], ignore_index=True)
    combined.to_excel(path, index=False, engine="openpyxl")
    logger.info(f"{chapter}: appended {len(ratio_rows):,} sex-ratio and "
                f"{len(share_rows):,} age-group row(s) -> {path.name} "
                f"({len(combined):,} rows)")
    return combined


In [ ]:
"""
CELL: Run both calculations on every chapter's combined English file.
"""
for chapter in chapters_to_process():
    print(f"\n=== {chapter} ===")
    add_calculations(chapter)

# The calculations above invent indicator titles and age-group labels that no
# questionnaire contains. Notebook 3 can only translate them if the dictionary
# has been taught them, and nothing else in the run would ever notice.
NEW_LABELS = check_calculated_labels()
print()
if NEW_LABELS.empty:
    print("The dictionary knows every label these calculations invent.")
else:
    print(f"{len(NEW_LABELS)} label(s) invented here are NOT in the dictionary:")
    for _, row in NEW_LABELS.iterrows():
        print(f"   [{row['col_en']}] {row['val_en']}")
    print("\nThese will come out in English from notebook 3 until they are added.")
    print("Run export_calculated_labels(), have Claude Code fill in val_ar,")
    print("then update_dictionary() - or just ask it to fill the dictionary gaps.")



## Gaps

`export_untranslated(REPORTS)` writes every value the dictionary could not
translate to `untranslated_values.xlsx`, with `val_en` blank. Ask Claude Code to
fill it in, then `update_dictionary()` writes the rows back into
`translation dict.xlsx` and the next run translates them.


In [ ]:
"""
CELL: Write the gap file for the run above.
"""
UNTRANSLATED = export_untranslated(REPORTS)
UNTRANSLATED.head(20)
